In [1]:
import zipfile, os

extract_dir = '/home/vsu/Downloads/cifar100_data'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile('/home/vsu/Downloads/archive_cifar100.zip', 'r') as z:
    z.extractall(extract_dir)
import zipfile, os

extract_dir = '/home/vsu/Downloads/cifar100_data'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile('/home/vsu/Downloads/archive_cifar100.zip', 'r') as z:
    z.extractall(extract_dir)

In [2]:
!mkdir /home/vsu/Downloads/cifar100_data/cifar-100-python
!mv /home/vsu/Downloads/cifar100_data/train \
   /home/vsu/Downloads/cifar100_data/test \
   /home/vsu/Downloads/cifar100_data/meta \
   /home/vsu/Downloads/cifar100_data/file.txt \
   /home/vsu/Downloads/cifar100_data/cifar-100-python/

mkdir: cannot create directory ‘/home/vsu/Downloads/cifar100_data/cifar-100-python’: File exists


In [3]:
!find /home/vsu/Downloads/cifar100_data -maxdepth 3

/home/vsu/Downloads/cifar100_data
/home/vsu/Downloads/cifar100_data/cifar-100-python
/home/vsu/Downloads/cifar100_data/cifar-100-python/file.txt
/home/vsu/Downloads/cifar100_data/cifar-100-python/test
/home/vsu/Downloads/cifar100_data/cifar-100-python/meta
/home/vsu/Downloads/cifar100_data/cifar-100-python/train


In [4]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset, ConcatDataset


# ==============================================================================
# 1. CIFAR-style ResNet
#    (Standard torchvision ResNet-18/34 are designed for 224x224 ImageNet
#     images and downsample too aggressively for 32x32 CIFAR images. We use
#     the widely-used "CIFAR ResNet" stem instead: 3x3 conv, stride 1, no
#     initial max-pool.)
# ==============================================================================
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class ResNetCIFAR(nn.Module):
    """ResNet adapted for 32x32 CIFAR images.
    Stem is a single 3x3 conv (stride 1, no maxpool) instead of the
    ImageNet 7x7-stride-2 + maxpool stem, since CIFAR images are much
    smaller and aggressive early downsampling destroys too much signal.
    """

    def __init__(self, block, num_blocks, num_classes=100):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out


def ResNet18(num_classes=100):
    # ResNet-18 = BasicBlock, [2, 2, 2, 2] blocks per stage
    return ResNetCIFAR(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)


def ResNet34(num_classes=100):
    # ResNet-34 = BasicBlock, [3, 4, 6, 3] blocks per stage
    return ResNetCIFAR(BasicBlock, [3, 4, 6, 3], num_classes=num_classes)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.backends.cudnn.benchmark = True  # fixed 32x32 input size -> free speedup

# ---- Runtime tuning ported from the fast AlexNet ZO-PGA pipeline ----------
# (Copy_of_Alexnet_ZOPGA_pipeline_fast.py: configure_fast_mode / set_tf32)
# Same idea: pick worker/precision settings from the hardware actually
# available, don't touch anything that affects the training math.
import os as _os
_cpu_count = _os.cpu_count() or 2
NUM_WORKERS = max(0, min(16, _cpu_count - 1))  # matches make_loader()'s cap in the fast pipeline
PREFETCH_FACTOR = 2 if NUM_WORKERS <= 2 else (4 if NUM_WORKERS <= 8 else 6)
BF16_OK = device.type == "cuda" and torch.cuda.is_bf16_supported()
torch.set_num_threads(max(1, min(_cpu_count, 32)))
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"NUM_WORKERS={NUM_WORKERS} PREFETCH_FACTOR={PREFETCH_FACTOR} BF16_OK={BF16_OK}")

#!pip install -q --force-reinstall "sympy==1.13.3"

import sympy
import sympy.printing
import torchvision
import torchvision.transforms as T

CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_STD = (0.2675, 0.2565, 0.2761)
# 2. Data Augmentation and Loading
# Using standard CIFAR-10 normalization constants
transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

transform_test = T.Compose([
    T.ToTensor()
])

@torch.no_grad()
def evaluate(model, device, loader):
  model.eval()
  correct, total=0.0, 0.0
  for x, y in loader:
    x,y=x.to(device), y.to(device)
    pred=torch.argmax(model(x), dim=1)
    correct+= (pred==y).sum().item()
    total+=y.size(0)
  return 100.0 * correct / total



test_set = torchvision.datasets.CIFAR100(root='/home/vsu/Downloads/cifar100_data', train=False, download=False, transform=transform_test)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
train_set = torchvision.datasets.CIFAR100(root='/home/vsu/Downloads/cifar100_data', train=True, download=False, transform=transform_train)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

Device: cuda
NUM_WORKERS=16 PREFETCH_FACTOR=6 BF16_OK=True


In [5]:
############ Normalization ##############
def normalize(x, mean, std):
    mean = torch.tensor(mean, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    std = torch.tensor(std, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    return (x - mean) / std

def unnorm(x,mean=CIFAR_MEAN, std=CIFAR_STD):
    mean = torch.tensor(mean, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    std = torch.tensor(std, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    return x*std+mean

class normalization(nn.Module):
  def __init__(self,backbone):
    super().__init__()
    self.backbone=backbone
  def forward(self,x):
    y=normalize(x, CIFAR_MEAN, CIFAR_STD)
    return self.backbone(y)

In [3]:
# # ==============================================================================
# # standard supervised teacher training on real CIFAR-10.
# # ==============================================================================
# # ==============================================================================
# # standard supervised teacher training on real CIFAR-10, saving best model on the fly.
# # ==============================================================================
# def train_teacher(teacher, train_loader, test_loader, device, epochs=100, lr=0.1,
#                    ckpt_path='/home/vsu/Downloads/ResNet34_cifar100.pth'):
#     teacher.to(device)
#     opt = torch.optim.SGD(teacher.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
#     sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
#     ce = nn.CrossEntropyLoss()

#     best_acc = 0.0

#     for epoch in range(epochs):
#         teacher.train()
#         for x, y in train_loader:
#             x, y = x.to(device), y.to(device)
#             opt.zero_grad()
#             loss = ce(teacher(x), y)
#             loss.backward()
#             opt.step()
#         sched.step()

#         if test_loader is not None:
#             acc = evaluate(teacher, device, test_loader)

#             if (epoch + 1) % 10 == 0:
#                 print(f"[teacher] epoch {epoch + 1}/{epochs}  test_acc={acc:.2f}%  best_acc={best_acc:.2f}%")

#             if acc > best_acc:
#                 best_acc = acc
#                 torch.save(teacher.state_dict(), ckpt_path)
#                 print(f"[teacher] epoch {epoch + 1}/{epochs}  new best acc={acc:.2f}%  -> saved to {ckpt_path}")

#     print(f"[teacher] training done. best_acc={best_acc:.2f}%")
#     return teacher, best_acc


# teacher_backbone = ResNet34().to(device)
# teacher = normalization(teacher_backbone).to(device)

# teacher_ckpt_path = '/home/vsu/Downloads/ResNet34_cifar100.pth'
# teacher, best_acc = train_teacher(
#     teacher, train_loader, test_loader, device,
#     epochs=100, lr=0.01, ckpt_path=teacher_ckpt_path
# )

# print(f"Best teacher checkpoint saved -> {teacher_ckpt_path} (test_acc={best_acc:.2f}%)")

In [6]:
########### Initial Noise #########################
# --- Noise initializer for PGA seeding --------------------------------------
def noise_smooth_gradient(n, device, size=32):
    """Smooth linear gradient at a random angle between two random colours."""
    yy, xx = torch.meshgrid(torch.linspace(0, 1, size), torch.linspace(0, 1, size), indexing="ij")
    imgs = torch.zeros(n, 3, size, size)
    for i in range(n):
        angle = random.uniform(0, 2 * math.pi)
        grad = xx * math.cos(angle) + yy * math.sin(angle)
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)  # kept on CPU so DataLoader workers can use it

def _perlin_grid(size, res, device):
    """Single 2D Perlin field, values roughly in [-1, 1]. `size` must be divisible by `res`."""
    assert size % res == 0, "size must be divisible by res"
    d = size // res
    lin = torch.arange(0, res, 1.0 / d)
    gy, gx = torch.meshgrid(lin, lin, indexing="ij")
    grid = torch.stack((gy % 1, gx % 1), dim=-1)             # (size, size, 2)

    angles = 2 * math.pi * torch.rand(res + 1, res + 1)
    grads = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)
    tile = lambda g: g.repeat_interleave(d, 0).repeat_interleave(d, 1)

    g00, g10, g01, g11 = tile(grads[:-1, :-1]), tile(grads[1:, :-1]), tile(grads[:-1, 1:]), tile(grads[1:, 1:])
    n00 = (torch.stack((grid[..., 0],     grid[..., 1]),     -1) * g00).sum(-1)
    n10 = (torch.stack((grid[..., 0] - 1, grid[..., 1]),     -1) * g10).sum(-1)
    n01 = (torch.stack((grid[..., 0],     grid[..., 1] - 1), -1) * g01).sum(-1)
    n11 = (torch.stack((grid[..., 0] - 1, grid[..., 1] - 1), -1) * g11).sum(-1)

    fade = lambda t: 6 * t**5 - 15 * t**4 + 10 * t**3
    t = fade(grid)
    nx0 = torch.lerp(n00, n10, t[..., 0])
    nx1 = torch.lerp(n01, n11, t[..., 0])
    return torch.lerp(nx0, nx1, t[..., 1])


def noise_perlin(n, device, size=32):
    """Perlin noise field per image, colourised the same way as the gradient noises above."""
    imgs = torch.zeros(n, 3, size, size)
    res_options = [2, 4, 8]  # coarse/medium/fine cell grids, all divide 32 evenly
    for i in range(n):
        res = random.choice(res_options)
        grad = _perlin_grid(size, res, device)
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)


def noise_uniform(n, device, size=32):
    """Plain i.i.d. uniform noise — no spatial smoothness, unlike the gradient/perlin noises."""
    return torch.rand(n, 3, size, size).clamp(0.0, 1.0)

def noise_gabor(n, device, size=32):
    """Gaussian-windowed sinusoidal grating (Gabor patch), random orientation/frequency/phase."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing="ij")
    for i in range(n):
        theta = random.uniform(0, math.pi)
        freq = random.uniform(2.0, 8.0)
        phase = random.uniform(0, 2 * math.pi)
        sigma = random.uniform(0.3, 0.8)
        x_theta = xx * math.cos(theta) + yy * math.sin(theta)
        y_theta = -xx * math.sin(theta) + yy * math.cos(theta)
        gaussian = torch.exp(-(x_theta**2 + y_theta**2) / (2 * sigma**2))
        grating = torch.cos(2 * math.pi * freq * x_theta + phase)
        grad = gaussian * grating
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)


def noise_checkerboard(n, device, size=32):
    """Checkerboard with random cell size and random phase offset per image."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    for i in range(n):
        block = random.choice([2, 4, 8, 16])
        ox, oy = random.randint(0, block - 1), random.randint(0, block - 1)
        pattern = (((xx + ox) // block) + ((yy + oy) // block)) % 2
        grad = pattern.float()
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)

NOISE_INITIALIZERS = {
    "noise_smooth": noise_smooth_gradient,
    #"noise_smooth_radial": noise_radial_gradient,
    "noise_perlin": noise_perlin,
    "noise_uniform": noise_uniform,
    "noise_gabor": noise_gabor,
    "noise_checkerboard": noise_checkerboard,
}
NOISE_TYPES = list(NOISE_INITIALIZERS.keys())

########## KL DIvergence ################
def klpga(x, student, t_logits,T):
  s_logits=student(x)/T
  t_logits=t_logits.detach()
  log_prob= F.log_softmax(s_logits, dim=-1)
  t_prob=F.softmax(t_logits/T, dim=-1)
  kl_dv=F.kl_div(log_prob, t_prob, reduction="batchmean")*T**2
  return kl_dv

############Custom Dataset ####################
class CustomDataset(Dataset):
  def __init__(self,x):
    self.x=x
    # self.y=y
    # self.z=z
  def __len__(self):
    return(self.x.size(0))
  def __getitem__(self,idx):
    return self.x[idx]


########################## Dataset to Loader #################
# def data_to_loader(data):
#   n = len(data)
#   batch_size = n if n < 512 else 512
#   num_workers = 2 if n >= 512 else 0
#   return DataLoader(
#       data,
#       batch_size=batch_size,
#       shuffle=True,
#       num_workers=num_workers,
#       pin_memory=True,
#       persistent_workers=num_workers > 0,
#       #multiprocessing_context="spawn" if num_workers > 0 else None,
#   )

# Cache one DataLoader per batch size so the (intentional) 16 -> 512 batch-size
# ramp doesn't respawn the worker pool every single epoch. The ramp itself is
# unchanged; only the loader lifecycle is optimized.
#
# num_workers / prefetch_factor / drop_last below are ported from
# make_loader() in Copy_of_Alexnet_ZOPGA_pipeline_fast.py (the pipeline where
# student training runs fast) -- same settings, same reasoning, no change to
# what gets fed to the loss.
_loader_cache = {}

_loader_cache = {}

#-----------------------------------------------------------------
def data_to_loader(data, i):
    if i < 25:
        batch_size = 16
    elif i<50:
        batch_size=64
    elif i<75:
        batch_size=128
    elif i<100:
        batch_size=256
    elif i<125:
        batch_size=512
    elif i<150:
        batch_size=1024
    else:
      batch_size=2048

#--------------------------------------------------------------------------
#                               65%
#-----------------------------------------------------------------------------
    # def data_to_loader(data, i):
    # if i < 25:
    #     batch_size = 16
    # elif i<50:
    #     batch_size=64
    # elif i<75:
    #     batch_size=128
    # elif i<100:
    #     batch_size=256
    # elif i<125:
    #     batch_size=512
    # elif i<150:
    #     batch_size=1024
    # else:
    #   batch_size=2048
#----------------------------------------------------------------------------------
#----------------------------------------------------------------------------------------------

    if batch_size not in _loader_cache:
        num_workers = NUM_WORKERS  # hardware-aware, set up in the setup cell
        kwargs = dict(
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True,
            drop_last=True,   # keeps batch shape constant -> cudnn.benchmark stays hot
        )
        if num_workers > 0:
            kwargs["persistent_workers"] = True
            kwargs["prefetch_factor"] = PREFETCH_FACTOR
            #kwargs["multiprocessing_context"] = "spawn"
        _loader_cache[batch_size] = DataLoader(data, **kwargs)
    return _loader_cache[batch_size]



In [7]:
# =============================================================================
# Augmentation
#
#   Two INDEPENDENT switches, applied in this order:
#     1. geometric base  : RandomCrop(32, pad=4, reflect) + RandomHorizontalFlip
#     2. random op stack : k ops sampled without replacement from the 17-op pool
#
#   Both stages happen inside Dataset.__getitem__, i.e. BEFORE the tensor is
#   handed to the teacher. So in Phase 3 and Phase 5 the teacher is queried on
#   the crop+flip view as well as the op-stack view (see AUG_QUERY_NOTE).
# =============================================================================
base_geo_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

_perspective_tf = T.RandomPerspective(distortion_scale=0.35, p=1.0)
_zoom_crop_tf = T.RandomResizedCrop(32, scale=(0.65, 1.0), ratio=(0.85, 1.15))
_color_jitter_tf = T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.45, hue=0.12)
_cutout_tf = T.RandomErasing(p=1.0, scale=(0.02, 0.25), ratio=(0.3, 3.3), value=0.0)


def op_rotate(img):
    return T.functional.rotate(img, random.uniform(-20, 20))


def op_affine(img):
    t = 2
    return T.functional.affine(
        img,
        angle=random.uniform(-12, 12),
        translate=(random.randint(-t, t), random.randint(-t, t)),
        scale=random.uniform(0.82, 1.18),
        shear=random.uniform(-12, 12),
    )


def op_perspective(img):
    return _perspective_tf(img)


def op_zoom_crop(img):
    return _zoom_crop_tf(img)


def op_color_jitter(img):
    return _color_jitter_tf(img)


def op_grayscale(img):
    return T.functional.rgb_to_grayscale(img, num_output_channels=3)


def op_gaussian_blur(img):
    k = random.choice([3, 5])
    return T.functional.gaussian_blur(img, kernel_size=k, sigma=random.uniform(0.1, 2.2))


def op_sharpness(img):
    return T.functional.adjust_sharpness(img, random.uniform(0.0, 3.5))


def op_autocontrast(img):
    return T.functional.autocontrast(img.clamp(0.0, 1.0))


def op_equalize(img):
    img_u8 = (img.clamp(0.0, 1.0) * 255).to(torch.uint8)
    return T.functional.equalize(img_u8).float() / 255.0


def op_posterize(img):
    img_u8 = (img.clamp(0.0, 1.0) * 255).to(torch.uint8)
    return T.functional.posterize(img_u8, random.choice([3, 4, 5, 6])).float() / 255.0


def op_solarize(img):
    return T.functional.solarize(img.clamp(0.0, 1.0), random.uniform(0.3, 0.9))


def op_invert(img):
    return T.functional.invert(img.clamp(0.0, 1.0))


def op_gaussian_noise(img):
    sigma = random.uniform(0.01, 0.08)
    return (img + torch.randn_like(img) * sigma).clamp(0.0, 1.0)


def op_salt_pepper(img):
    prob = random.uniform(0.01, 0.06)
    mask = torch.rand(1, img.shape[1], img.shape[2], device=img.device)
    salt = (mask < prob / 2).expand_as(img)
    pepper = (mask > 1 - prob / 2).expand_as(img)
    out = img.clone()
    out[salt] = 1.0
    out[pepper] = 0.0
    return out


def op_cutout(img):
    return _cutout_tf(img.unsqueeze(0)).squeeze(0)


def op_random_color_erase(img):
    out = img.clone()
    h, w = img.shape[1], img.shape[2]
    eh, ew = random.randint(4, 12), random.randint(4, 12)
    y0 = random.randint(0, h - eh)
    x0 = random.randint(0, w - ew)
    out[:, y0:y0 + eh, x0:x0 + ew] = torch.rand(3, 1, 1, device=img.device)
    return out


# The 17-op pool. Identical membership, order and parameters in both pipelines.
AUG_OPS = {
    "rotate": op_rotate,
    "affine": op_affine,
    "perspective": op_perspective,
    "zoom_crop": op_zoom_crop,
    "color_jitter": op_color_jitter,
    "grayscale": op_grayscale,
    "gaussian_blur": op_gaussian_blur,
    "sharpness": op_sharpness,
    "autocontrast": op_autocontrast,
    "equalize": op_equalize,
    "posterize": op_posterize,
    "solarize": op_solarize,
    "invert": op_invert,
    "gaussian_noise": op_gaussian_noise,
    "salt_pepper": op_salt_pepper,
    "cutout": op_cutout,
    "random_color_erase": op_random_color_erase,
}
AUG_OP_NAMES = list(AUG_OPS.keys())
N_AUG_OPS = len(AUG_OP_NAMES)
assert N_AUG_OPS == 17, f"expected a 17-op pool, found {N_AUG_OPS}"

AUG_QUERY_NOTE = (
    "Teacher is queried on the AUGMENTED view (geo + ops), so crop/flip is "
    "inside the query path. Use --no-query_aug to distil against the logits "
    "stored at generation time instead."
)


def resolve_op_pool(enable=None, disable=None):
    """Restrict the 17-op pool. `enable`/`disable` are comma-separated name lists."""
    names = list(AUG_OP_NAMES)
    if enable:
        wanted = [s.strip() for s in enable.split(",") if s.strip()]
        unknown = [w for w in wanted if w not in AUG_OPS]
        if unknown:
            raise ValueError(f"unknown aug ops: {unknown}. Valid: {AUG_OP_NAMES}")
        names = [n for n in names if n in wanted]
    if disable:
        drop = {s.strip() for s in disable.split(",") if s.strip()}
        unknown = [d for d in drop if d not in AUG_OPS]
        if unknown:
            raise ValueError(f"unknown aug ops: {unknown}. Valid: {AUG_OP_NAMES}")
        names = [n for n in names if n not in drop]
    return names


def diverse_augment(img, use_geo=True, n_random_ops=4):
    """
    Independent switches:
      use_geo      -> apply RandomCrop(pad 4, reflect) + RandomHorizontalFlip
      n_random_ops -> how many ops to sample (without replacement) from op_pool
    Setting n_random_ops=0 with use_geo=True gives crop+flip only.
    Setting use_geo=False with n_random_ops=k gives the op stack only.
    """
    img = img.clamp(0.0, 1.0)
    if use_geo:
        img = base_geo_transform(img)
        pool = AUG_OP_NAMES
        k = min(int(n_random_ops), len(pool))
        for name in random.sample(pool, k=k):
            img = AUG_OPS[name](img)
            img = img.clamp(0.0, 1.0)
    return img


# =============================================================================
# Datasets
# =============================================================================
# class AugmentedDataset(Dataset):
#     """
#     Wraps a (img, label) base dataset. Yields augmented (img, label).
#     Used for the real-CIFAR-10 phases (4 and 5).
#     """

#     def __init__(self, base_ds, use_geo=True, n_random_ops=0, op_pool=None):
#         self.base_ds = base_ds
#         self.use_geo = use_geo
#         self.n_random_ops = n_random_ops
#         self.op_pool = op_pool

#     def __len__(self):
#         return len(self.base_ds)

#     def __getitem__(self, idx):
#         img, label = self.base_ds[idx]
#         img = diverse_augment(img, self.use_geo, self.n_random_ops, self.op_pool)
#         return img, label


class SyntheticDataset(Dataset):
    """Yields augmented (img, stored_label, stored_logits) for Phase 3."""

    def __init__(self, imgs, use_geo=True, n_random_ops=4):
        self.imgs = imgs
        self.use_geo=use_geo
        self.n_random_ops=n_random_ops

    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, idx):
        img = diverse_augment(self.imgs[idx], self.use_geo, self.n_random_ops)
        return img


In [8]:
############ Data from Scratch ##############
def noise_data(p):
    all_imgs = []
    for i in range(p):
        noise_fn = random.choice([noise_smooth_gradient, noise_perlin, noise_uniform, noise_gabor, noise_checkerboard])
        imgs = noise_fn(100, device, size=32)
        all_imgs.append(imgs)
    return torch.cat(all_imgs, dim=0)

In [9]:
################ Teacher and Student Models ##############################


teacher_backbone= ResNet34().to(device)
student_backbone=ResNet18().to(device)
teacher=normalization(teacher_backbone)
student=normalization(student_backbone)
teacher_path='/home/vsu/Downloads/ResNet34_cifar100.pth'
#checkpoint = torch.load(teacher_path, map_location=device)
# state_dict = checkpoint  # or checkpoint["state_dict"] / checkpoint["model"] if nested
# new_state_dict = {}
# for k, v in state_dict.items():
#     new_state_dict[f"backbone.{k}"] = v

# teacher.load_state_dict(new_state_dict)
teacher.load_state_dict(torch.load(teacher_path, map_location=device))
teacher.eval()
for p in teacher.parameters():
  p.requires_grad_(False)
evaluate(teacher, device, test_loader)

76.39

In [10]:
import math

def batch_aligned_lr(epoch, total_epochs=100):
    """Cosine decay within each phase, resetting to 0.01 at every batch-size
       change (epochs 20, 30, 45, 50, 60, 70, 80) to line up with the batch-size ramp."""
    if epoch < 25:
        start, end, phase_start, phase_len = 0.01, 0.00001, 0, 25
    elif epoch < 50:
          start, end, phase_start, phase_len = 0.001, 0.00001, 25, 25
    elif epoch < 75:
         start, end, phase_start, phase_len = 0.01, 0.00001, 50, 25
    elif epoch < 100:
         start, end, phase_start, phase_len = 0.01, 0.00001, 75, 25
    elif epoch < 125:
         start, end, phase_start, phase_len = 0.01, 0.00001, 100, 25
    elif epoch < 150:
         start, end, phase_start, phase_len = 0.01, 0.00001, 125, 25
    # elif epoch < 95:
    #     start, end, phase_start, phase_len = 0.001, 0.0001, 90, 95
    else:
        start, end, phase_start, phase_len = 0.01, 0.00001, 150, 50

    t = epoch - phase_start
    cos_factor = 0.5 * (1 + math.cos(math.pi * t / max(phase_len - 1, 1)))
    lr = end + (start - end) * cos_factor
    return lr / 0.01  # normalized multiplier for LambdaLR

#------------------------------ --------------------------------------------------------
#                                  65%
#-----------------------------------------------------------------------------------------------
# def batch_aligned_lr(epoch, total_epochs=100):
#     """Cosine decay within each phase, resetting to 0.01 at every batch-size
#        change (epochs 20, 30, 45, 50, 60, 70, 80) to line up with the batch-size ramp."""
#     if epoch < 25:
#         start, end, phase_start, phase_len = 0.01, 0.00001, 0, 25
#     elif epoch < 50:
#           start, end, phase_start, phase_len = 0.001, 0.00001, 25, 25
#     elif epoch < 75:
#          start, end, phase_start, phase_len = 0.01, 0.00001, 50, 25
#     elif epoch < 100:
#          start, end, phase_start, phase_len = 0.01, 0.00001, 75, 25
#     elif epoch < 125:
#          start, end, phase_start, phase_len = 0.01, 0.00001, 100, 25
#     elif epoch < 150:
#          start, end, phase_start, phase_len = 0.01, 0.00001, 125, 25
#     # elif epoch < 95:
#     #     start, end, phase_start, phase_len = 0.001, 0.0001, 90, 95
#     else:
#         start, end, phase_start, phase_len = 0.01, 0.00001, 150, 50

#     t = epoch - phase_start
#     cos_factor = 0.5 * (1 + math.cos(math.pi * t / max(phase_len - 1, 1)))
#     lr = end + (start - end) * cos_factor
#     return lr / 0.01  # normalized multiplier for LambdaLR
#------------------------------------------------------------------------------------------------------------    
#------------------------------------------------------------------------------------------------------

# def three_phase_lr(epoch):
#     """Cosine decay within each phase, resetting to 0.01 at epoch 20 and epoch 100
#        to line up with the batch-size ramp."""
#     if epoch < 20:
#         start, end, phase_start, phase_len = 0.01, 0.001, 0, 20
#     elif epoch < 100:
#         start, end, phase_start, phase_len = 0.01, 0.001, 20, 80
#     else:
#         start, end, phase_start, phase_len = 0.01, 0.0001, 100, 100  # assumes 200 total epochs

#     t = epoch - phase_start
#     cos_factor = 0.5 * (1 + math.cos(math.pi * t / max(phase_len - 1, 1)))
#     lr = end + (start - end) * cos_factor
#     return lr / 0.01

In [11]:
####################################################################
# Student Training
##################################################################
# ---- fast-pipeline optimizations below, ported from train_student_synth() /
# prepare_model_for_fast() / fast_input() / train_autocast() in
# Copy_of_Alexnet_ZOPGA_pipeline_fast.py (the script where student training
# is fast). Loss, optimizer, LR schedule, data and epoch count are unchanged.

def _fused_sgd_kwargs(device):
    # Fused CUDA kernel for the optimizer step; same math, same update rule.
    import inspect
    if device.type == "cuda" and "fused" in inspect.signature(torch.optim.SGD.__init__).parameters:
        return {"fused": True}
    return {}

def _prepare_for_fast(model, device):
    # channels_last is a pure memory-layout change (NHWC vs NCHW); conv math
    # is identical but cudnn kernels for it run faster on modern GPUs.
    if device.type == "cuda":
        return model.to(memory_format=torch.channels_last)
    return model

def _fast_input(x, device):
    if device.type == "cuda" and x.dim() == 4:
        return x.contiguous(memory_format=torch.channels_last)
    return x

def _train_autocast(device):
    # bf16 when the GPU supports it (no GradScaler needed -- bf16 has fp32's
    # exponent range so it doesn't underflow like fp16). Falls back to fp16
    # autocast+scaler on older GPUs (e.g. T4), or plain fp32 on CPU.
    if device.type == "cuda" and BF16_OK:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16), False
    if device.type == "cuda":
        return torch.cuda.amp.autocast(), True
    from contextlib import nullcontext
    return nullcontext(), False


def train_student(teacher, student, dataset, test_loader, device,T, student_epochs=100, lr=0.001):
  student = _prepare_for_fast(student, device)
  teacher = _prepare_for_fast(teacher, device)
  opt = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4,
                         **_fused_sgd_kwargs(device))
  sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=batch_aligned_lr)

#   sched = torch.optim.lr_scheduler.CosineAnnealingLR(
#     opt,
#     T_max=100,       # number of epochs to reach eta_min
#     eta_min=1e-6    # minimum learning rate
# )
  autocast_ctx, needs_scaler = _train_autocast(device)
  scaler = torch.cuda.amp.GradScaler(enabled=needs_scaler)
  for epoch in range(student_epochs):
    student.train()
    loader=data_to_loader(dataset,epoch)
    epoch_loss = 0.0
    for x in loader:
      x = _fast_input(x.to(device, non_blocking=True), device)
      with torch.no_grad(), autocast_ctx:  # teacher params already frozen; no_grad
        z = teacher(x)                     # also stops autograd tracking the input side
      opt.zero_grad(set_to_none=True)
      with autocast_ctx:
        loss = klpga(x, student, z, T)
      if needs_scaler:
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
      else:
        loss.backward()
        opt.step()
      epoch_loss += loss.item()
    sched.step()
    accuracy=evaluate(student, device, test_loader)
    print(accuracy)
    if (epoch + 1) % 25 == 0 or epoch == student_epochs - 1:
      print(f"epoch {epoch+1}/{student_epochs}  loss={epoch_loss/len(loader):.4f}")

  return student


In [12]:
data=noise_data(1500)
len(data)
print(data.shape)

torch.Size([150000, 3, 32, 32])


In [14]:
dataset= SyntheticDataset(data, use_geo=True, n_random_ops=8)


In [ ]:
train_student(teacher, student, dataset,test_loader, device,T=20, student_epochs=200, lr=0.01)

/tmp/ipykernel_5270/1701019884.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=needs_scaler)


7.93
13.03
15.29
16.29
16.44
20.37
21.68
21.71
23.5
